# ResNet-18 Transfer Learning trên CIFAR-10

## Mục tiêu

Notebook thực hiện hai thí nghiệm Transfer Learning với ResNet-18 pretrained trên ImageNet:

- **ResNet-18 A – FC only:** đóng băng toàn bộ backbone và chỉ huấn luyện lớp `fc`.
- **ResNet-18 B – Layer4 + FC:** fine-tuning `layer4` và lớp `fc`.

Hai thí nghiệm sử dụng chung:
- CIFAR-10
- DataLoader chung của nhóm
- Training engine chung của nhóm
- Optimizer Adam
- Learning rate = 0.001
- Batch size = 32
- 5 epochs

Mục tiêu là so sánh chiến lược chỉ huấn luyện classifier cuối với chiến lược fine-tuning thêm `layer4`.

In [1]:
from pathlib import Path
import sys
import time

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt


# Tìm thư mục gốc Practice_2
def find_project_root():
    current = Path.cwd().resolve()

    candidates = [
        current,
        current.parent,
        current.parent.parent,
    ]

    for candidate in candidates:
        if (
            (candidate / "src").exists()
            and (candidate / "configs").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc Practice_2."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Import code dùng chung của nhóm
from src.data import DataConfig, build_dataloaders
from src.trainer import get_device, train_model
from src.models.resnet18 import (
    build_resnet18_fc_only,
    build_resnet18_layer4_fc,
    count_trainable_parameters,
)


device = get_device()

print("===== PROJECT SETUP =====")
print("PROJECT_ROOT :", PROJECT_ROOT)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())
print("Device       :", device)

===== PROJECT SETUP =====
PROJECT_ROOT : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2
PyTorch      : 2.13.0+cpu
CUDA         : False
Device       : cpu


## 3. DataLoader dùng chung

Sử dụng pipeline dữ liệu chung của nhóm từ `src.data`.

Dataset được đọc trực tiếp từ:

`Practice_2/data/raw`

Không tạo lại dataset hoặc tự chia dữ liệu trong notebook này.

In [5]:
# ============================================================
# CELL 3 - SHARED DATALOADERS
# ============================================================

# Đường dẫn dữ liệu dùng chung của nhóm
data_dir = PROJECT_ROOT / "data" / "raw"

# Cấu hình dữ liệu
data_config = DataConfig(
    data_dir=data_dir,
    image_size=224,
    batch_size=32,
    val_ratio=0.10,
    num_workers=0,
    seed=42
)

# build_dataloaders() trả về một dictionary
data_bundle = build_dataloaders(data_config)

# Lấy các DataLoader và thông tin cần thiết từ dictionary
train_loader = data_bundle["train_loader"]
val_loader = data_bundle["val_loader"]
test_loader = data_bundle["test_loader"]
class_names = data_bundle["class_names"]

train_dataset = data_bundle["train_dataset"]
val_dataset = data_bundle["val_dataset"]
test_dataset = data_bundle["test_dataset"]

print("===== DATALOADER READY =====")
print("Data directory :", data_dir)
print("Train samples  :", len(train_dataset))
print("Val samples    :", len(val_dataset))
print("Test samples   :", len(test_dataset))
print("Train batches  :", len(train_loader))
print("Val batches    :", len(val_loader))
print("Test batches   :", len(test_loader))
print("Classes        :", class_names)
print("Number classes :", len(class_names))

===== DATALOADER READY =====
Data directory : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\data\raw
Train samples  : 45000
Val samples    : 5000
Test samples   : 10000
Train batches  : 1407
Val batches    : 157
Test batches   : 313
Classes        : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Number classes : 10


## 4. Xây dựng ResNet-18 A và ResNet-18 B

- **ResNet-18 A:** chỉ huấn luyện lớp `fc`.
- **ResNet-18 B:** fine-tuning `layer4` và `fc`.

Cả hai mô hình sử dụng ResNet-18 pretrained trên ImageNet và thay lớp phân loại cuối để phù hợp với 10 lớp CIFAR-10.

In [7]:
# ============================================================
# CELL 4 - BUILD RESNET-18 A AND B
# ============================================================

NUM_CLASSES = len(class_names)

model_a = build_resnet18_fc_only(
    num_classes=NUM_CLASSES
)

model_b = build_resnet18_layer4_fc(
    num_classes=NUM_CLASSES
)

print("===== RESNET-18 MODELS CREATED =====")
print("Number of classes :", NUM_CLASSES)

print("\nResNet-18 A")
print("Strategy           : FC only")
print("Final FC layer     :", model_a.fc)

print("\nResNet-18 B")
print("Strategy           : Layer4 + FC")
print("Final FC layer     :", model_b.fc)

===== RESNET-18 MODELS CREATED =====
Number of classes : 10

ResNet-18 A
Strategy           : FC only
Final FC layer     : Linear(in_features=512, out_features=10, bias=True)

ResNet-18 B
Strategy           : Layer4 + FC
Final FC layer     : Linear(in_features=512, out_features=10, bias=True)


## 5. So sánh số tham số có thể huấn luyện

Kiểm tra số lượng tham số trainable của hai chiến lược để xác nhận việc freeze/fine-tuning được thiết lập đúng.

In [8]:
# ============================================================
# CELL 5 - TRAINABLE PARAMETERS
# ============================================================

total_a = sum(
    p.numel()
    for p in model_a.parameters()
)

total_b = sum(
    p.numel()
    for p in model_b.parameters()
)

trainable_a = count_trainable_parameters(model_a)
trainable_b = count_trainable_parameters(model_b)

frozen_a = total_a - trainable_a
frozen_b = total_b - trainable_b

print("===== PARAMETER COMPARISON =====")

print("\nResNet-18 A - FC only")
print(f"Total params     : {total_a:,}")
print(f"Trainable params : {trainable_a:,}")
print(f"Frozen params    : {frozen_a:,}")

print("\nResNet-18 B - Layer4 + FC")
print(f"Total params     : {total_b:,}")
print(f"Trainable params : {trainable_b:,}")
print(f"Frozen params    : {frozen_b:,}")

===== PARAMETER COMPARISON =====

ResNet-18 A - FC only
Total params     : 11,181,642
Trainable params : 5,130
Frozen params    : 11,176,512

ResNet-18 B - Layer4 + FC
Total params     : 11,181,642
Trainable params : 8,398,858
Frozen params    : 2,782,784


## 6. Forward-pass check

Kiểm tra một batch dữ liệu thật từ DataLoader chung để đảm bảo:

- dữ liệu được đưa lên đúng `device`,
- cả hai mô hình chạy được,
- output có đúng 10 lớp CIFAR-10.

In [9]:
# ============================================================
# CELL 6 - FORWARD PASS CHECK
# ============================================================

device = get_device()

# Lấy một batch thật từ DataLoader chung
images, labels = next(iter(train_loader))

# Đưa model lên device
model_a = model_a.to(device)
model_b = model_b.to(device)

# Đưa dữ liệu lên cùng device
images = images.to(device)
labels = labels.to(device)

# Chuyển sang evaluation mode để kiểm tra forward pass
model_a.eval()
model_b.eval()

with torch.no_grad():
    outputs_a = model_a(images)
    outputs_b = model_b(images)

print("===== FORWARD PASS CHECK =====")
print("Device         :", device)
print("Images device  :", images.device)
print("Labels device  :", labels.device)

print("\nInput shape    :", images.shape)
print("Labels shape   :", labels.shape)
print("Output A shape :", outputs_a.shape)
print("Output B shape :", outputs_b.shape)

===== FORWARD PASS CHECK =====
Device         : cpu
Images device  : cpu
Labels device  : cpu

Input shape    : torch.Size([32, 3, 224, 224])
Labels shape   : torch.Size([32])
Output A shape : torch.Size([32, 10])
Output B shape : torch.Size([32, 10])


## 7. Cấu hình training chung

Hai thí nghiệm sử dụng cùng cấu hình huấn luyện để đảm bảo so sánh công bằng:

- Loss: `CrossEntropyLoss`
- Optimizer: `Adam`
- Learning rate: `0.001`
- Batch size: `32`
- Epochs: `5`
- Device: tự động lấy từ `get_device()`

Checkpoint và TensorBoard log được lưu trong thư mục dùng chung của `Practice_2`.

In [10]:
# ============================================================
# CELL 7 - SHARED TRAINING CONFIG
# ============================================================

LEARNING_RATE = 0.001
NUM_EPOCHS = 5

# Thư mục dùng chung
checkpoint_dir = PROJECT_ROOT / "checkpoints"
runs_dir = PROJECT_ROOT / "runs"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)

# Loss function dùng chung
criterion = nn.CrossEntropyLoss()

# Optimizer A - chỉ lấy các tham số trainable
optimizer_a = optim.Adam(
    filter(lambda p: p.requires_grad, model_a.parameters()),
    lr=LEARNING_RATE
)

# Optimizer B - chỉ lấy các tham số trainable
optimizer_b = optim.Adam(
    filter(lambda p: p.requires_grad, model_b.parameters()),
    lr=LEARNING_RATE
)

# Đường dẫn checkpoint
checkpoint_a = checkpoint_dir / "resnet18_fc_only_best.pth"
checkpoint_b = checkpoint_dir / "resnet18_layer4_fc_best.pth"

# TensorBoard log directory
log_dir_a = runs_dir / "resnet18_fc_only"
log_dir_b = runs_dir / "resnet18_layer4_fc"

device = get_device()

print("===== TRAINING CONFIG =====")
print("Loss          : CrossEntropyLoss")
print("Optimizer A   : Adam")
print("Optimizer B   : Adam")
print("Learning rate :", LEARNING_RATE)
print("Batch size    :", data_config.batch_size)
print("Epochs        :", NUM_EPOCHS)
print("Device        :", device)

print("\nCheckpoint A  :", checkpoint_a)
print("Checkpoint B  :", checkpoint_b)

print("\nTensorBoard A :", log_dir_a)
print("TensorBoard B :", log_dir_b)

===== TRAINING CONFIG =====
Loss          : CrossEntropyLoss
Optimizer A   : Adam
Optimizer B   : Adam
Learning rate : 0.001
Batch size    : 32
Epochs        : 5
Device        : cpu

Checkpoint A  : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\checkpoints\resnet18_fc_only_best.pth
Checkpoint B  : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\checkpoints\resnet18_layer4_fc_best.pth

TensorBoard A : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\runs\resnet18_fc_only
TensorBoard B : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\runs\resnet18_layer4_fc
